# Add a New CNN Style to the Gallery

Use this notebook for **NST TransformerNet** models trained with `kaggle_multi_pic_trainer.ipynb`.
These models always use the `nchw` tensor layout — no configuration needed.

Run cells **1 → 5** in order.  
Cell 3 opens a file-picker dialog; cell 4 shows model file info; cell 5 takes the style name and description; cell 6 writes everything to disk.

---

### Model file formats

| File | What it is | Needed at… |
|---|---|---|
| `.onnx` | Open Neural Network Exchange. Self-contained inference graph **+ embedded weights**. Framework-agnostic — runs on ONNX Runtime without PyTorch. | Runtime (app uses this) |
| `.onnx.data` | External weight blob produced automatically when the model exceeds ~2 GB (protobuf limit). The `.onnx` file stores the graph + references; the `.data` file stores the actual bytes. **Both files must stay in the same folder.** | Runtime (alongside `.onnx`) |
| `.pth` | PyTorch state-dict (pickle). Raw weight tensors — kept as a backup for future fine-tuning or re-export only. | Training / re-export only |

**For GAN models** (CycleGAN, AnimeGAN) use `add_GAN_style.ipynb` instead.


In [1]:
import sys, pathlib

# -- make scripts/ importable
_scripts_dir = pathlib.Path(".").resolve()
if str(_scripts_dir) not in sys.path:
    sys.path.insert(0, str(_scripts_dir))

from add_style_helper import setup

ctx = setup()


Repository   : C:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer
Existing styles: ['abstract', 'anime', 'candy', 'escher', 'hundertwasser', 'mosaic', 'mucha', 'muchaworks', 'rain_princess', 'starry_night']


In [2]:
from add_style_helper import pick_onnx_file

paths = pick_onnx_file()


.onnx : C:\Users\i09300076\Downloads\model.onnx
.pth  : C:\Users\i09300076\Downloads\model.pth


In [3]:
from add_style_helper import report_model_files

report_model_files(paths.onnx_path, paths.pth_path, paths.data_path)


  .onnx      OK  (6.8 MB)
  .pth       OK  (6.7 MB)
  .onnx.data -- not present (weights embedded in .onnx)


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from add_style_helper import validate_style_id

name_w = widgets.Text(
    placeholder="e.g. Rain Princess",
    description="Style name:",
    layout=widgets.Layout(width="380px"),
)
desc_w = widgets.Text(
    placeholder="e.g. Rainy impressionist style",
    description="Description:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "90px"},
)
author_w = widgets.Text(
    value="PeterWazinski",
    description="Author:",
    layout=widgets.Layout(width="380px"),
)
status_out = widgets.Output()

def _check(_: object = None) -> None:
    with status_out:
        status_out.clear_output(wait=True)
        raw = name_w.value.strip()
        if not raw:
            return
        _, msg = validate_style_id(raw, ctx.existing_ids)
        print(f"{msg}  tensor_layout='nchw'")

name_w.observe(_check, names="value")
display(widgets.VBox([name_w, desc_w, author_w, status_out]))


In [ ]:
from add_style_helper import install_style

style_id = install_style(
    onnx_path=paths.onnx_path,
    pth_path=paths.pth_path,
    data_path=paths.data_path,
    style_name=name_w.value,
    style_desc=desc_w.value,
    style_author=author_w.value,
    tensor_layout="nchw",
    styles_dir=ctx.styles_dir,
    catalog_path=ctx.catalog_path,
    catalog=ctx.catalog,
    existing_ids=ctx.existing_ids,
    content_image=ctx.content_image,
    repo_root=ctx.repo_root,
)

from IPython.display import display, Image as IPyImage
preview_path = ctx.styles_dir / style_id / "preview.jpg"
if preview_path.exists():
    print(f"\nPreview thumbnail:")
    display(IPyImage(filename=str(preview_path), width=256))


Copied model.pth
Preview generated: C:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer\styles\tintin\preview.jpg

OK  'Tintin'  (id='tintin', layout='nchw')  added to gallery.
Files:
  model.onnx  (6.8 MB)
  model.pth  (6.7 MB)
  preview.jpg  (0.0 MB)

Next: git add -A ; git commit -m 'feat: add <name> style'
